# データベース演習 第13回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### データベースをダウンロード

* 全データが含まれています
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります
* その度にダウンロードするのが面倒な人は，自分のGoogleドライブに保存し，そのファイルを参照する方法もあります（生成AIなどで調べてみてください）．

In [ ]:
# Google DriveからSQLiteファイルを取得
import gdown

# ファイルIDを指定
file_id = '1nuCOXMd3H0rL-82D460SnbpM7YjLNBeq'
url = f'https://drive.google.com/uc?id={file_id}'

# 保存ファイル名を指定
output = 'weblog.sqlite3'
gdown.download(url, output, quiet=False)

### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化`
%load_ext sql

In [ ]:
# 結果表示数は以下の数値を変えれば変更できる
%config SqlMagic.displaylimit = 10 # デフォルトは10行

### Webログデータベース

In [ ]:
# Webログデータベースに接続する
%sql sqlite:///weblog.sqlite3

### 含まれるテーブルの確認

In [ ]:
%%sql
SELECT * FROM sqlite_master;


### 例題1

customersテーブルは顧客の情報である，顧客のID（customer_idカラム），年齢（customer_ageカラム）などを管理している．顧客の年齢にギャップがどの程度あるかを調べたい．年齢のギャップとは，年齢が小さい順に顧客を並べたときの隣接する2人の顧客の年齢の差を意味することとする．LAGウィンドウ関数を用いることにより，顧客のID，顧客の年齢，年齢のギャップ（AS句を用いage_gapという名前にせよ）の組を，age_gapの大きい順に並べ，最大10行のみ求めるSQL文を作成せよ

In [ ]:
%%sql
SELECT customer_id, customer_age, customer_age- LAG(customer_age) OVER (
	ORDER BY customer_age
) AS age_gap
FROM customers
ORDER BY age_gap DESC LIMIT 10;

## 例題2

顧客を表すcustomersテーブルを用いて，氏名（customer_nameカラム）と，年齢（customer_ageカラム）その人が属する年齢グループ（別名をage_groupとせよ）の組を，最大10行，返却するSQL文を作成せよ．年齢グループの値は，30歳未満：若者，30歳以上50歳未満：中年，50歳以上：老人，という値をとることとする．

In [ ]:
%%sql
SELECT customer_name, customer_age,
	CASE
		WHEN customer_age < 30 THEN '若者'
		WHEN customer_age < 50 THEN '中年'
		ELSE '老人'
	END AS age_group
FROM customers LIMIT 10;

## 演習 課題6-3

itemsテーブルは商品の情報である，商品のID（item_idカラム），名前（item_nameカラム）価格（item_priceカラム）などを管理している．商品の価格にギャップがどの程度あるかを調べたい．価格のギャップとは，価格が安い順（安い商品→高い商品の順）に商品を並べたときの隣接する2つの商品の価格の差を意味することとする．LAGウィンドウ関数を用いることにより，商品のID，商品の名前，商品の価格，価格のギャップ（AS句を用いprice_gapという名前にせよ）の組を，price_gapの大きい順に並べ，最大10行のみ求めるSQL文を作成せよ

In [ ]:
%%sql


### 演習 課題6-4

顧客を表すcustomersテーブルを用いて，顧客のID（customer_idカラム），氏名（customer_nameカラム）と，居住地（customer_locationカラム）その人が新潟県に住んでいるかどうかのフラグ（別名をniigata_flagとせよ）の組を，最大10行，返却するSQL文を作成せよ．niigata_flagの値は，居住地が新潟県である場合に1をとり，それ以外の都道府県である場合は0をとることとする．

In [ ]:
%%sql
